# 06. Теория вероятностей

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/06_probability.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m06). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**6.2.** Проверьте моделированием: при какой дисперсии весов сигнал не затухает через 50 слоёв ReLU (ширина 512)?

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
x0 = np.random.randn(1000, 512)
for name, std in [("0.01", 0.01), ("Xavier 1/n", np.sqrt(1/512)), ("He 2/n", np.sqrt(2/512))]:
    x = x0
    for _ in range(50):
        x = np.maximum(0, x @ (np.random.randn(512, 512) * std))
    print(f"{name:12s} std активаций после 50 слоёв: {x.std():.2e}")
# 0.01 → ~0, Xavier → затухает в ~2^25 раз, He → порядка 1

### 🏋️ Практика модуля

**6.3.** Оцените π методом Монте-Карло и найдите, сколько точек нужно для точности 0.001.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
u = np.random.rand(1_000_000, 2)
pi_hat = 4 * ((u**2).sum(1) < 1).mean()

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def distributions():
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    k = np.arange(0, 15)
    axes[0].bar(k, stats.binom.pmf(k, 14, 0.3), color=C[0]); axes[0].set_title("Биномиальное B(n=14, p=0.3)")
    axes[1].bar(k, stats.poisson.pmf(k, 3), color=C[2]); axes[1].set_title("Пуассон (λ=3)")
    x = np.linspace(-5, 5, 400)
    for (mu, s), c in zip([(0, 1), (0, 2), (1.5, 0.6)], C):
        axes[2].plot(x, stats.norm.pdf(x, mu, s), color=c, lw=2, label=f"μ={mu}, σ={s}")
    axes[2].legend(fontsize=8); axes[2].set_title("Нормальное N(μ, σ²)")
    x = np.linspace(0, 1, 400)
    for (a, b), c in zip([(0.5, 0.5), (2, 5), (5, 2), (2, 2)], C):
        axes[3].plot(x, stats.beta.pdf(x, a, b), color=c, lw=2, label=f"α={a}, β={b}")
    axes[3].set_ylim(0, 3.5); axes[3].legend(fontsize=8); axes[3].set_title("Бета Beta(α, β) — «распределение над p»")
    save(fig, "distributions")

distributions()

In [ ]:
def mvn():
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    X, Y = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
    pos = np.dstack([X, Y])
    for ax, (S, t) in zip(axes, [([[1, 0], [0, 1]], "Σ = I: независимые, «круг»"),
                                 ([[2, 0], [0, 0.4]], "диагональная Σ: разные дисперсии"),
                                 ([[1, 0.8], [0.8, 1]], "ρ = 0.8: корреляция наклоняет эллипс")]):
        ax.contourf(X, Y, stats.multivariate_normal([0, 0], S).pdf(pos), levels=12, cmap="Blues")
        ax.set_aspect("equal"); ax.set_title(t, fontsize=10)
    save(fig, "mvn")

mvn()

In [ ]:
def clt():
    rng = np.random.default_rng(0)
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
    for ax, n in zip(axes, [1, 2, 5, 30]):
        means = rng.exponential(1.0, size=(20000, n)).mean(1)
        ax.hist(means, bins=60, density=True, color=C[0], alpha=0.6)
        x = np.linspace(means.min(), means.max(), 200)
        ax.plot(x, stats.norm.pdf(x, 1, 1 / np.sqrt(n)), color=C[1], lw=2)
        ax.set_title(f"среднее {n} экспоненциальных")
    fig.suptitle("ЦПТ: среднее независимых величин → нормальное N(μ, σ²/n), даже если исходное распределение кривое", y=1.05)
    save(fig, "clt")

clt()